In [ ]:
import os
import sys

import numpy as np
import polars as pl
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

sys.path.append("/workspace")

from drsd.reader import MINDsmallReader

pl.Config(tbl_rows=5)
os.makedirs("/workspace/processed", exist_ok=True)

In [ ]:
reader = MINDsmallReader()

In [ ]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

### Target News

In [ ]:
news_df = (
    pl.concat([reader.get_news_df("train"), reader.get_news_df("dev")])
    .select("news_id", "title", "abstract")
    .unique()
    .sort("news_id")
)
assert news_df.get_column("news_id").is_unique().all()
news_df

### Extract News Embedding

In [ ]:
embedding = model.encode(
    [
        "\n".join(
            [f"[title] {news_d['title']}"]
            + ([f"[abstract] {news_d['abstract']}"] if news_d["abstract"] is not None else [])
        )
        for news_d in news_df.to_dicts()
    ],
)
embedding

### PCA

In [ ]:
pca = PCA(n_components=16, random_state=0)
assert type(embedding) is np.ndarray
embedding_pca = pca.fit_transform(embedding)
embedding_pca

### Output DataFrame

In [ ]:
output_df = news_df.select(
    "news_id", pl.Series(embedding).alias("embedding"), pl.Series(embedding_pca).alias("embedding_pca")
)
output_df.write_parquet("/workspace/processed/news_embedding.parquet")
output_df